# Лабораторная работа 14

## Скрапинг страницы сотрудников РГПУ им. А. И. Герцена

**Выполнил:** Киселев Георгий Петрович

### Задание

1. Со страниц `https://atlas.herzen.spb.ru/teachers?page=1` — `https://atlas.herzen.spb.ru/teachers?page=54` получить список преподавателей: ФИО и ссылку на профиль.
2. Со страницы профиля загрузить почту и телефон при наличии.
3. Сформировать CSV-файл.
4. Реализовать решение двумя способами:
   - `BeautifulSoup`;
   - альтернативная библиотека `lxml`.


## 1. Установка и импорт библиотек

Если библиотек нет, сначала выполнить:

```bash
pip install requests beautifulsoup4 lxml
```


In [ ]:
from __future__ import annotations

import csv
import random
import re
import time
from dataclasses import dataclass
from typing import Iterable
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from lxml import html as lxml_html
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://atlas.herzen.spb.ru"
LIST_URL = BASE_URL + "/teachers?page={page}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; educational-scraper/1.0)",
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
}

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-zА-Яа-я]{2,}")
PHONE_RE = re.compile(
    r"(?:(?:\+7|8)\s*[\(\- ]?\s*\d{3}\s*[\)\- ]?\s*\d{3}\s*[\- ]?\s*\d{2}\s*[\- ]?\s*\d{2}"
    r"(?:\s*(?:доб\.?|добавочный|ext\.?)\s*\d{1,6})?)"
)
TEACHER_URL_RE = re.compile(r"/teachers/(\d+)(?:$|[?#])")

CSV_COLUMNS = ["ФИО", "Почта", "Телефон", "Ссылка на профиль"]

START_PAGE = 1
END_PAGE = 54
DELAY = 0.2


## 2. Общие функции

Они используются в обоих вариантах: создают HTTP-сессию, нормализуют текст, удаляют дубли и сохраняют CSV.


In [ ]:
@dataclass(frozen=True)
class Teacher:
    fio: str
    profile_url: str
    page: int | None = None


@dataclass(frozen=True)
class TeacherContacts:
    fio: str
    email: str
    phone: str
    profile_url: str


def make_session() -> requests.Session:
    """Создаёт requests-сессию с повторными попытками при временных ошибках."""
    session = requests.Session()
    session.headers.update(HEADERS)

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def get_html(session: requests.Session, url: str, timeout: int = 25) -> str:
    """Загружает HTML-страницу и возвращает текст."""
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    if not response.encoding:
        response.encoding = response.apparent_encoding
    return response.text


def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def unique_keep_order(values: Iterable[str]) -> list[str]:
    """Удаляет дубли, сохраняя порядок."""
    result = []
    seen = set()
    for value in values:
        value = value.strip(" ,;\n\t")
        if value and value not in seen:
            result.append(value)
            seen.add(value)
    return result


def save_csv(rows: list[TeacherContacts], output_path: str) -> None:
    """Сохраняет результат в CSV с кодировкой utf-8-sig для корректного открытия в Excel."""
    with open(output_path, "w", encoding="utf-8-sig", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {
                    "ФИО": row.fio,
                    "Почта": row.email,
                    "Телефон": row.phone,
                    "Ссылка на профиль": row.profile_url,
                }
            )


def print_stats(rows: list[TeacherContacts], filename: str) -> None:
    with_email = sum(bool(row.email) for row in rows)
    with_phone = sum(bool(row.phone) for row in rows)
    print(f"Файл сохранён: {filename}")
    print(f"Всего строк: {len(rows)}")
    print(f"Строк с почтой: {with_email}")
    print(f"Строк с телефоном: {with_phone}")


## 3. Способ 1 — BeautifulSoup

В этом варианте HTML разбирается через `BeautifulSoup`.


In [ ]:
def bs4_parse_teachers_from_list_page(html: str, page: int) -> list[Teacher]:
    """Извлекает преподавателей с одной страницы списка через BeautifulSoup."""
    soup = BeautifulSoup(html, "html.parser")
    teachers = []
    seen_urls = set()

    # На странице могут быть и табличное представление, и карточки.
    # Поэтому ищем все ссылки на /teachers/<id> и удаляем дубли.
    for link in soup.select('a[href*="/teachers/"]'):
        href = link.get("href")
        fio = normalize_spaces(link.get_text(" ", strip=True))

        if not href or not fio:
            continue

        profile_url = urljoin(BASE_URL, href)

        if not TEACHER_URL_RE.search(profile_url):
            continue
        if profile_url in seen_urls:
            continue
        if len(fio.split()) < 2:
            continue

        teachers.append(Teacher(fio=fio, profile_url=profile_url, page=page))
        seen_urls.add(profile_url)

    return teachers


def bs4_collect_teachers(session: requests.Session, start_page: int = 1, end_page: int = 54, delay: float = 0.2) -> list[Teacher]:
    """Собирает ФИО и ссылки на профили со страниц списка."""
    all_teachers = []
    seen_urls = set()

    for page in range(start_page, end_page + 1):
        url = LIST_URL.format(page=page)
        html = get_html(session, url)
        page_teachers = bs4_parse_teachers_from_list_page(html, page)

        added = 0
        for teacher in page_teachers:
            if teacher.profile_url not in seen_urls:
                all_teachers.append(teacher)
                seen_urls.add(teacher.profile_url)
                added += 1

        print(f"Страница {page}: найдено {len(page_teachers)}, добавлено {added}, всего {len(all_teachers)}")
        time.sleep(delay + random.random() * delay)

    return all_teachers


def bs4_parse_contacts_from_profile(html: str) -> tuple[str, str]:
    """Извлекает почту и телефон из профиля через BeautifulSoup."""
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(" ", strip=True)
    emails = unique_keep_order(EMAIL_RE.findall(text))
    phones = unique_keep_order(PHONE_RE.findall(text))

    return "; ".join(emails), "; ".join(phones)


def bs4_collect_contacts(session: requests.Session, teachers: list[Teacher], delay: float = 0.2) -> list[TeacherContacts]:
    """Загружает профили и формирует строки для CSV."""
    rows = []
    total = len(teachers)

    for index, teacher in enumerate(teachers, start=1):
        try:
            html = get_html(session, teacher.profile_url)
            email, phone = bs4_parse_contacts_from_profile(html)
        except Exception as exc:
            print(f"Ошибка профиля {teacher.profile_url}: {exc}")
            email, phone = "", ""

        rows.append(TeacherContacts(teacher.fio, email, phone, teacher.profile_url))

        if index % 25 == 0 or index == total:
            print(f"Профили: {index}/{total}")
        time.sleep(delay + random.random() * delay)

    return rows


### Запуск BeautifulSoup-варианта

Эта ячейка создаёт файл `teachers_contacts_bs4.csv`.


In [ ]:
session = make_session()

bs4_teachers = bs4_collect_teachers(session, START_PAGE, END_PAGE, DELAY)
bs4_rows = bs4_collect_contacts(session, bs4_teachers, DELAY)

bs4_output = "teachers_contacts_bs4.csv"
save_csv(bs4_rows, bs4_output)
print_stats(bs4_rows, bs4_output)


## 4. Способ 2 — lxml

Во втором варианте используется библиотека `lxml`. По логике решение аналогично первому варианту, но извлечение данных выполняется через XPath.


In [ ]:
def lxml_parse_teachers_from_list_page(html: str, page: int) -> list[Teacher]:
    """Извлекает преподавателей с одной страницы списка через lxml + XPath."""
    doc = lxml_html.fromstring(html)
    teachers = []
    seen_urls = set()

    for link in doc.xpath('//a[contains(@href, "/teachers/")]'):
        href = link.get("href")
        fio = normalize_spaces(" ".join(link.xpath(".//text()")))

        if not href or not fio:
            continue

        profile_url = urljoin(BASE_URL, href)

        if not TEACHER_URL_RE.search(profile_url):
            continue
        if profile_url in seen_urls:
            continue
        if len(fio.split()) < 2:
            continue

        teachers.append(Teacher(fio=fio, profile_url=profile_url, page=page))
        seen_urls.add(profile_url)

    return teachers


def lxml_collect_teachers(session: requests.Session, start_page: int = 1, end_page: int = 54, delay: float = 0.2) -> list[Teacher]:
    """Собирает ФИО и ссылки на профили со страниц списка через lxml."""
    all_teachers = []
    seen_urls = set()

    for page in range(start_page, end_page + 1):
        url = LIST_URL.format(page=page)
        html = get_html(session, url)
        page_teachers = lxml_parse_teachers_from_list_page(html, page)

        added = 0
        for teacher in page_teachers:
            if teacher.profile_url not in seen_urls:
                all_teachers.append(teacher)
                seen_urls.add(teacher.profile_url)
                added += 1

        print(f"Страница {page}: найдено {len(page_teachers)}, добавлено {added}, всего {len(all_teachers)}")
        time.sleep(delay + random.random() * delay)

    return all_teachers


def lxml_parse_contacts_from_profile(html: str) -> tuple[str, str]:
    """Извлекает почту и телефон из профиля через lxml."""
    doc = lxml_html.fromstring(html)

    for bad_node in doc.xpath("//script|//style|//noscript"):
        parent = bad_node.getparent()
        if parent is not None:
            parent.remove(bad_node)

    text = " ".join(part.strip() for part in doc.xpath("//body//text()") if part.strip())
    emails = unique_keep_order(EMAIL_RE.findall(text))
    phones = unique_keep_order(PHONE_RE.findall(text))

    return "; ".join(emails), "; ".join(phones)


def lxml_collect_contacts(session: requests.Session, teachers: list[Teacher], delay: float = 0.2) -> list[TeacherContacts]:
    """Загружает профили и формирует строки для CSV через lxml."""
    rows = []
    total = len(teachers)

    for index, teacher in enumerate(teachers, start=1):
        try:
            html = get_html(session, teacher.profile_url)
            email, phone = lxml_parse_contacts_from_profile(html)
        except Exception as exc:
            print(f"Ошибка профиля {teacher.profile_url}: {exc}")
            email, phone = "", ""

        rows.append(TeacherContacts(teacher.fio, email, phone, teacher.profile_url))

        if index % 25 == 0 or index == total:
            print(f"Профили: {index}/{total}")
        time.sleep(delay + random.random() * delay)

    return rows


### Запуск lxml-варианта

Эта ячейка создаёт файл `teachers_contacts_lxml.csv`. Если не нужно повторно нагружать сайт, можно не запускать эту ячейку после BeautifulSoup-варианта.


In [ ]:
session = make_session()

lxml_teachers = lxml_collect_teachers(session, START_PAGE, END_PAGE, DELAY)
lxml_rows = lxml_collect_contacts(session, lxml_teachers, DELAY)

lxml_output = "teachers_contacts_lxml.csv"
save_csv(lxml_rows, lxml_output)
print_stats(lxml_rows, lxml_output)


## 5. Проверка результата

После выполнения одного из вариантов можно быстро посмотреть первые строки CSV.


In [ ]:
import pandas as pd

# Можно заменить имя файла на teachers_contacts_lxml.csv
result = pd.read_csv("teachers_contacts_bs4.csv")
result.head(10)


## Вывод

В ходе работы был выполнен скрапинг страниц преподавателей РГПУ им. А. И. Герцена. Сначала со страниц списка были получены ФИО и ссылки на профили, затем из профилей были извлечены почты и телефоны при наличии. Итоговые данные сохраняются в CSV-файл.

Реализованы два варианта решения: с использованием `BeautifulSoup` и с использованием `lxml`. Оба варианта дают одинаковую структуру итогового CSV.
